# Adaptive Prompting Engine for Medical Chatbot

This notebook implements a Phase 4 prompting layer that wraps the fine-tuned medical assistant. It enforces structured clinical safety prompts, adapts tone and detail dynamically, tracks conversation context, and performs self-checks before responding.

In [42]:
from __future__ import annotations

import json
from pathlib import Path

from app.phase5 import adaptive_prompt_engine as engine_module
from app.phase5.adaptive_prompt_engine import AdaptivePromptEngine, PatientProfile

In [43]:
core_configuration = {
    "stress_keywords": sorted(engine_module.STRESS_KEYWORDS),
    "general_follow_up_questions": engine_module.GENERAL_FOLLOW_UP_QUESTIONS,
    "red_flag_categories": engine_module.RED_FLAG_CATEGORY_MAP,
}
core_configuration

{'stress_keywords': ['afraid',
  'anxious',
  'panic',
  'scared',
  'terrified',
  'urgent',
  'worried'],
 'general_follow_up_questions': ['Do you have chest pain, shortness of breath, or difficulty swallowing?',
  'What medications or self-care steps have you already tried today?',
  'Have your symptoms changed in severity or pattern compared with yesterday?'],
 'red_flag_categories': {'chest': ('Cardiac or pulmonary emergency',
   'Chest pain or tightness can signal a heart attack or lung emergency and needs ambulance-level care.'),
  'shortness of breath': ('Respiratory distress',
   'Trouble breathing may worsen quickly—ensure immediate medical evaluation.'),
  'difficulty swallowing': ('Airway compromise',
   'Difficulty swallowing can mean airway swelling; urgent in-person assessment is required.'),
  'cannot breathe': ('Airway compromise',
   'Inability to breathe is a medical emergency—activate emergency services now.'),
  'fever over': ('Systemic infection concern',
   'High

In [44]:
PatientProfile, engine_module.ConversationState

(app.phase5.adaptive_prompt_engine.PatientProfile,
 app.phase5.adaptive_prompt_engine.ConversationState)

In [45]:
helper_functions = [
    engine_module.detect_tone,
    engine_module.detect_detail_preference,
    engine_module.generate_follow_up_questions,
    engine_module.generate_clinical_reasoning,
    engine_module.generate_progressive_summary
]
helper_functions

[<function app.phase5.adaptive_prompt_engine.detect_tone(text: 'str') -> 'str'>,
 <function app.phase5.adaptive_prompt_engine.detect_detail_preference(text: 'str') -> 'str'>,
 <function app.phase5.adaptive_prompt_engine.generate_follow_up_questions(state: 'ConversationState', new_symptoms: 'List[str]') -> 'List[str]'>,
 <function app.phase5.adaptive_prompt_engine.generate_clinical_reasoning(state: 'ConversationState', profile: 'PatientProfile', new_red_flags: 'List[str]') -> 'List[str]'>,
 <function app.phase5.adaptive_prompt_engine.generate_progressive_summary(intent: 'str', urgent: 'bool', new_symptoms: 'List[str]', known_symptoms: 'List[str]', trigger: 'Optional[str]') -> 'str'>]

In [46]:
engine_module.STRUCTURED_PROMPT_TEMPLATE.splitlines()[:20]

['',
 'You are a clinical support assistant helping a remote triage nurse interpret patient updates. Use the structured context below to craft a safe, empathetic response.',
 '',
 '[Patient Context]',
 '{patient_context}',
 '',
 '[Conversation Summary]',
 '{conversation_summary}',
 '',
 '[Current Symptoms]',
 '{symptom_summary}',
 '',
 '[Emotional State]',
 '{emotional_summary}',
 '',
 '[New Information Detected]',
 '{new_information}',
 '',
 '[Red Flags Detected]',
 '{red_flags}']

In [47]:
# Example usage with doctor-style reasoning and follow-up prompts
patient_profile = PatientProfile(
    age="45",
    gender="female",
    history=["Type 2 diabetes", "Hypertension"],
    medications=["Metformin", "Lisinopril"],
)

engine = AdaptivePromptEngine(patient_profile=patient_profile)

# Simulate prior conversation to seed context
engine.ingest_patient_message("I have been feeling dizzy and nauseous for two days.")
engine.ingest_assistant_message(
    "Can you share when the dizziness began, whether it comes with chest pain, and what makes it better or worse?"
)

latest_patient_input = "I'm scared because the dizziness is worse and now I feel chest tightness."
result = engine.build_prompt(latest_patient_input)

print(result["prompt"][:600] + "\n\n...")  # truncated for readability
metadata = result["metadata"]
metadata_preview = {
    "progressive_summary": metadata.get("progressive_summary"),
    "clinical_reasoning": metadata.get("clinical_reasoning"),
    "follow_up_questions": metadata.get("follow_up_questions"),
    "red_flag_actions": metadata.get("red_flag_actions"),
    "urgent": metadata.get("urgent"),
    "intent": metadata.get("intent"),
}
print("\nMetadata preview:\n", json.dumps(metadata_preview, indent=2))


You are a clinical support assistant helping a remote triage nurse interpret patient updates. Use the structured context below to craft a safe, empathetic response.

[Patient Context]
Age: 45
Gender: female
Relevant history: Type 2 diabetes; Hypertension
Medications: Metformin; Lisinopril

[Conversation Summary]
Patient: I have been feeling dizzy and nauseous for two days.
Assistant: Can you share when the dizziness began, whether it comes with chest pain, and what makes it better or worse?
Patient: I'm scared because the dizziness is worse and now I feel chest tightness.

[Current Symptoms]


...

Metadata preview:
 {
  "progressive_summary": "Emergency warning triggered\u2014direct the patient to seek immediate in-person care.",
  "clinical_reasoning": [
    "Chest pain: Cardiac history raises worry that chest pain or breathlessness is heart-related\u2014escalate quickly.",
    "Cardiac or pulmonary emergency: Chest pain or tightness can signal a heart attack or lung emergency and n

### Integration notes

- Call `engine.build_prompt(patient_message)` before invoking the fine-tuned LLM. The returned `prompt` already encodes a progressive summary, reasoning cues, clarifying questions, emergency actions, and safety guardrails.
- Examine `metadata["follow_up_questions"]`, `metadata["clinical_reasoning"]`, and `metadata["red_flag_actions"]` to drive UI panels, clinician alerts, and evaluation metrics.
- After sending the model reply back to the user, call `engine.ingest_assistant_message(reply_text)` to keep the conversation state current.
- Periodically persist `engine.export_state(Path("expert_feedback/conversation_state.json"))` so human reviewers can inspect context and audit trails.
- Use `engine.integrate_correction_feedback(Path("expert_feedback/annotations.jsonl"))` to surface unsafe cases gathered during earlier phases and iterate on guardrails.
- Extend the keyword lists for tone, detail, and emergency detection with domain-specific vocabulary as you collect more feedback.